# Analysis grid

Build the analysis grid that the analysis will be mapped onto.

In [1]:
%load_ext autoreload
%autoreload 2

## Imports

In [4]:
from pathlib import Path

import numpy as np
import geopandas as gpd
from shapely.geometry import box
import yaml

## Configuration 

In [14]:
# Generic settings
with open("../configs/project_config.yaml") as f:
    gen_config = yaml.safe_load(f)
    
PROCESSED_DATA_DIR = Path(gen_config["data_dir"]["processed"]) # The data/processed/ should have no subdirs
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
CELL_SIZE = gen_config["grid_cell_size_deg"]

# Spacetime settings
with open("../configs/census_scope.yaml") as f:
    census_config = yaml.safe_load(f)

BOUNDING_BOX = tuple(census_config["bounding_box"].values())


## Step 1 - Make the grid

In [15]:
def make_grid(minx, miny, maxx, maxy, cell_size=0.05):
    xs = np.arange(minx, maxx, cell_size)
    ys = np.arange(miny, maxy, cell_size)
    cells = [box(x, y, x + cell_size, y + cell_size) for x in xs for y in ys]
    grid = gpd.GeoDataFrame({"geometry": cells}, crs="EPSG:4326")
    grid["cell_id"] = grid.index
    grid["centroid_lon"] = grid.geometry.centroid.x
    grid["centroid_lat"] = grid.geometry.centroid.y
    return grid

In [16]:
analysis_grid = make_grid(*BOUNDING_BOX, cell_size=CELL_SIZE)
analysis_grid.head()

/tmp/ipykernel_251908/1111188012.py:7: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  grid["centroid_lon"] = grid.geometry.centroid.x
/tmp/ipykernel_251908/1111188012.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  grid["centroid_lat"] = grid.geometry.centroid.y


,geometry,cell_id,centroid_lon,centroid_lat
0,"POLYGON ((60.4748 5.35352, 60.4748 5.40352, 60...",0,60.449805,5.378521
1,"POLYGON ((60.4748 5.40352, 60.4748 5.45352, 60...",1,60.449805,5.428521
2,"POLYGON ((60.4748 5.45352, 60.4748 5.50352, 60...",2,60.449805,5.478521
3,"POLYGON ((60.4748 5.50352, 60.4748 5.55352, 60...",3,60.449805,5.528521
4,"POLYGON ((60.4748 5.55352, 60.4748 5.60352, 60...",4,60.449805,5.578521


Safe to ignore the warning in this case

## Step 2 - Save grid

In [17]:
# Save grid to processed directory (it's final)
analysis_grid.to_parquet(PROCESSED_DATA_DIR / "grid.parquet")
